<a href="https://colab.research.google.com/github/PavloIeremeiew/BD_FIT_1-7m/blob/main/BD_Lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DateType
import pyspark.sql.functions as F
from google.colab import files
import os

import yfinance as yf
import pandas as pd

In [23]:
spark = SparkSession.builder.appName("Lab3").master("local[*]").getOrCreate()

In [24]:
data = [
    (("Ivanov", "Ivan"), 25, "M", [95, 85, 90]),
    (("Petrov", "Petro"), 30, "M", [70, 80, 75]),
    (("Koval", "Anna"), 22, "F", [100, 95, 98])
]

schema = StructType([
    StructField('name', StructType([
        StructField('lastname', StringType(), True),
        StructField('firstname', StringType(), True)
    ])),
    StructField('age', IntegerType(), True),
    StructField('gender', StringType(), True),
    StructField('scores', StringType(), True) # Для простоти покажемо як масив або рядок
])

df_manual = spark.createDataFrame(data=data, schema=schema)
df_manual.show()
df_manual.printSchema()
print("Використання collect():", df_manual.collect()[0])

+---------------+---+------+-------------+
|           name|age|gender|       scores|
+---------------+---+------+-------------+
| {Ivanov, Ivan}| 25|     M| [95, 85, 90]|
|{Petrov, Petro}| 30|     M| [70, 80, 75]|
|  {Koval, Anna}| 22|     F|[100, 95, 98]|
+---------------+---+------+-------------+

root
 |-- name: struct (nullable = true)
 |    |-- lastname: string (nullable = true)
 |    |-- firstname: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- scores: string (nullable = true)

Використання collect(): Row(name=Row(lastname='Ivanov', firstname='Ivan'), age=25, gender='M', scores='[95, 85, 90]')


In [5]:
# Завантажуємо файл"problem2_data set.txt",. Структура: stationId, timeOfTheReading, readingType, temperatureRecorded, ...
uploaded = files.upload()
file_name2 = list(uploaded.keys())[0]

df_temp = spark.read.csv(file_name2, inferSchema=True, header=False)
df_temp = df_temp.select(
    F.col("_c0").alias("stationId"),
    F.col("_c1").alias("timeOfTheReading"),
    F.col("_c2").alias("readingType"),
    F.col("_c3").alias("temperatureRecorded")
)
# Фільтруємо TMIN та знаходимо мінімум для кожної станції
df_tmin = df_temp.filter(F.col("readingType") == "TMIN")
df_tmin.groupBy("stationId").agg(F.min("temperatureRecorded").alias("Min_Temperature")).show()

Saving problem2_data set.txt to problem2_data set.txt
+-----------+---------------+
|  stationId|Min_Temperature|
+-----------+---------------+
|ITE00100554|           -148|
|EZE00100082|           -135|
+-----------+---------------+



In [6]:
# Читаємо як текстовий файл problem3_data set.txt
uploaded = files.upload()
file_name3 = list(uploaded.keys())[0]

df_text = spark.read.text(file_name3)

# Розбиваємо на слова, переводимо в нижній регістр, фільтруємо пусті
df_words = df_text.select(F.explode(F.split(F.lower(F.col("value")), "\\s+")).alias("word"))
df_words = df_words.filter(df_words.word != "")
# Рахуємо та виводимо топ-10
df_words.groupBy("word").count().orderBy(F.desc("count")).show(10)

Saving problem3_data set.txt to problem3_data set.txt
+---------+-----+
|     word|count|
+---------+-----+
|     data|  361|
|      big|  285|
|       in|  171|
| training|  114|
|   course|  105|
|   hadoop|  100|
|   online|   58|
|  courses|   53|
|    spark|   42|
|bangalore|   40|
+---------+-----+
only showing top 10 rows


In [7]:
# problem4_data set.txt
# Структура: CustomerId, ProductId, Amount
uploaded = files.upload()
file_name4 = list(uploaded.keys())[0]

df_sales = spark.read.csv(file_name4, inferSchema=True, header=False)
df_sales = df_sales.select(
    F.col("_c0").alias("CustomerId"),
    F.col("_c1").alias("ProductId"),
    F.col("_c2").alias("Amount")
)
# Групуємо за клієнтом та сумуємо витрати
df_top_customers = df_sales.groupBy("CustomerId").agg(F.sum("Amount").alias("Total_Spent"))
df_top_customers.orderBy(F.desc("Total_Spent")).show(10)

Saving problem4_data set.txt to problem4_data set.txt
+----------+-----------------+
|CustomerId|      Total_Spent|
+----------+-----------------+
|        68|6375.449999999997|
|        73|6206.199999999999|
|        39|6193.109999999999|
|        54|6065.389999999999|
|        71|5995.660000000003|
|         2|          5994.59|
|        97|5977.189999999995|
|        46|5963.109999999999|
|        42|5696.840000000003|
|        59|          5642.89|
+----------+-----------------+
only showing top 10 rows


In [8]:
# problem5_data set.txt
#Структура: userid, movieid, rating, timestamp (як числа/рядки розділені табуляцією)
uploaded = files.upload()
file_name5 = list(uploaded.keys())[0]

df_ratings = spark.read.option("delimiter", "\t").csv(file_name5, inferSchema=True, header=False)
df_ratings = df_ratings.select(
    F.col("_c0").alias("userid"),
    F.col("_c1").alias("movieid"),
    F.col("_c2").alias("rating"),
    F.col("_c3").alias("timestamp")
)
# Рахуємо кількість кожної з оцінок та сортуємо
df_ratings.groupBy("rating").count().orderBy(F.desc("rating")).show()

Saving problem5_data set.txt to problem5_data set.txt
+------+-----+
|rating|count|
+------+-----+
|     5|21201|
|     4|34174|
|     3|27145|
|     2|11370|
|     1| 6110|
+------+-----+



частина 2

In [9]:
# Завантажуємо історичні дані акцій Microsoft (MSFT) за 10 років
ticker = "MSFT"
print(f"Завантаження даних {ticker} з Yahoo Finance...")
pdf = yf.Ticker(ticker).history(period="10y")

# Очищення даних для створення Spark DataFrame (перетворюємо індекс з датою у звичайну колонку)
pdf.reset_index(inplace=True)
pdf['Date'] = pd.to_datetime(pdf['Date']).dt.date # Прибираємо часові пояси
# Конвертуємо Pandas DataFrame у Spark DataFrame
df_stock = spark.createDataFrame(pdf)

Завантаження даних MSFT з Yahoo Finance...


In [10]:
print("1. Перевірка схеми")
df_stock.printSchema()

1. Перевірка схеми
root
 |-- Date: date (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Dividends: double (nullable = true)
 |-- Stock Splits: double (nullable = true)



In [11]:
print("2. Перші 10 рядків")
df_stock.show(10)

2. Перші 10 рядків
+----------+------------------+------------------+------------------+------------------+--------+---------+------------+
|      Date|              Open|              High|               Low|             Close|  Volume|Dividends|Stock Splits|
+----------+------------------+------------------+------------------+------------------+--------+---------+------------+
|2016-10-28|  53.2869125277513| 53.73977771659314| 52.90508973127995|    53.16259765625|33574700|      0.0|         0.0|
|2016-10-31| 53.42011821180075|  53.6509882434896| 53.20700454711914| 53.20700454711914|26434700|      0.0|         0.0|
|2016-11-01| 53.25140099875808| 53.29579868707135| 52.61206344760223| 53.10044479370117|24533000|      0.0|         0.0|
|2016-11-02|53.118206903468966|53.215883854488204| 52.65646347240407| 52.77190017700195|22147000|      0.0|         0.0|
|2016-11-03| 52.86069115959883| 52.95836810249142|52.487746527455876|52.576541900634766|21600400|      0.0|         0.0|
|2016-11-04| 

In [12]:
print("3. Створення стовпця HV Ratio")
# HV Ratio = High / Volume
df_stock = df_stock.withColumn("HV Ratio", F.col("High") / F.col("Volume"))
df_stock.select("Date", "High", "Volume", "HV Ratio").show(5)

3. Створення стовпця HV Ratio
+----------+------------------+--------+--------------------+
|      Date|              High|  Volume|            HV Ratio|
+----------+------------------+--------+--------------------+
|2016-10-28| 53.73977771659314|33574700|1.600603362549572...|
|2016-10-31|  53.6509882434896|26434700|2.029566752922847...|
|2016-11-01| 53.29579868707135|24533000|2.172412615133548...|
|2016-11-02|53.215883854488204|22147000|2.402848415337888E-6|
|2016-11-03| 52.95836810249142|21600400|2.451730898617221E-6|
+----------+------------------+--------+--------------------+
only showing top 5 rows


In [13]:
print("4. У який день був пік ціни?")
# Знаходимо день з найвищою ціною High
max_price = df_stock.agg(F.max("High")).collect()[0][0]
df_stock.filter(F.col("High") == max_price).select("Date", "High").show()

4. У який день був пік ціни?
+----------+-----------------+
|      Date|             High|
+----------+-----------------+
|2026-10-13|551.1500244140625|
+----------+-----------------+



In [14]:
print("5. Середнє значення стовпця Close")
df_stock.agg(F.avg("Close").alias("Average_Close")).show()

5. Середнє значення стовпця Close
+------------------+
|     Average_Close|
+------------------+
|253.57789288789118|
+------------------+



In [15]:
print("6. Максимальне та мінімальне значення стовпця Об’єм (Volume)")
df_stock.agg(F.max("Volume").alias("Max_Volume"), F.min("Volume").alias("Min_Volume")).show()

6. Максимальне та мінімальне значення стовпця Об’єм (Volume)
+----------+----------+
|Max_Volume|Min_Volume|
+----------+----------+
| 186201600|   5855900|
+----------+----------+



In [16]:
print("7. Кількість днів, коли Close < 60 доларів")
days_below_60 = df_stock.filter(F.col("Close") < 60).count()
print(f"Кількість днів з ціною закриття менше $60: {days_below_60}")

7. Кількість днів, коли Close < 60 доларів
Кількість днів з ціною закриття менше $60: 120


In [17]:
print("8. Відсоток часу, коли High > 80 доларів")
total_days = df_stock.count()
days_above_80 = df_stock.filter(F.col("High") > 80).count()
percentage = (days_above_80 / total_days) * 100
print(f"Відсоток часу, коли максимальна ціна перевищувала $80: {round(percentage, 2)}%")

8. Відсоток часу, коли High > 80 доларів
Відсоток часу, коли максимальна ціна перевищувала $80: 88.18%


In [18]:
print("9. Щорічний максимум ціни High")
df_stock_years = df_stock.withColumn("Year", F.year("Date"))
df_stock_years.groupBy("Year").agg(F.max("High").alias("Max_High")).orderBy("Year").show()

9. Щорічний максимум ціни High
+----+------------------+
|Year|          Max_High|
+----+------------------+
|2016| 57.30322316263696|
|2017|  79.9686234736463|
|2018|107.55352352067055|
|2019|150.53476272282796|
|2020|221.45004902915983|
|2021| 336.2018030638584|
|2022|324.98135761149877|
|2023| 376.2672475889846|
|2024| 460.2370169877969|
|2025| 550.0131135040644|
|2026| 551.1500244140625|
+----+------------------+



In [19]:
print("10. Середнє закриття (Close) для кожного календарного місяця")
df_stock_months = df_stock.withColumn("Month", F.month("Date"))
df_stock_months.groupBy("Month").agg(F.avg("Close").alias("Average_Close")).orderBy("Month").show()

10. Середнє закриття (Close) для кожного календарного місяця
+-----+------------------+
|Month|     Average_Close|
+-----+------------------+
|    1|238.45625709195443|
|    2|239.14784194786512|
|    3|232.86812790126018|
|    4|242.22961981319688|
|    5|249.58868669114023|
|    6|255.31047985314183|
|    7| 270.0093158685936|
|    8|274.59331522807815|
|    9| 279.0521062589159|
|   10| 276.8163110715849|
|   11|236.44771989031537|
|   12|245.01694891897685|
+-----+------------------+



In [25]:
spark.stop()